In [5]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from ipywidgets import interactive, IntSlider
from IPython.display import Markdown, display, Latex



In [6]:
# Alleen train inladen
df = pd.read_csv("data_train.csv", sep=",", decimal=".")


FileNotFoundError: [Errno 2] No such file or directory: 'data_train.csv'

In [5]:
# Scheiden van target en input variabelen
target = "Churn"

X = df.drop(columns=[target])
y = df[target]

X.shape, y.shape

((1995, 9), (1995,))

In [6]:
# Er is maar een categorische variabelen waar we OneHotEncoder op kunnen gebruiken namelijk Age group, 
# ookal is dit eigenlijk niet nodig omdat we ook een numerieke Age variabelen in de dataset hebben, maar de opdracht zei dat het moest...
cat_cols = ["Age Group"]
num_cols = [c for c in X.columns if c not in cat_cols]

In [9]:
# ColumnTransformer voor OneHotEncoder en StandardScaler voor de overige numerieke variabelen
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [10]:
# Hier maken we de pipline aan 
prep_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])
prep_pipeline

,steps,"[('preprocessing', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
# Hier stoppen we X (input variabelen) in onze pipeline en leert de pipeline hoe hij deze moet schalen/encoden
X_prepared = prep_pipeline.fit_transform(X)
X_prepared.shape

(1995, 13)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

NameError: name 'churn' is not defined

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

NameError: name 'X' is not defined

In [ ]:
from sklearn.model_selection import GridSearchCV

mtry = [2,3,4,5,6,7,8,9,10,20,30]

gridsearch_rf = GridSearchCV(
    rf_pipe,
    param_grid={"model__max_features": mtry},
    scoring="roc_auc",
    n_jobs=-1
)

gridsearch_rf.fit(X_train, y_train)

In [ ]:
mtry = [2,3,4,5,6,7,8,9,10,20,30]

gridsearch_rf = GridSearchCV(
    rf_pipe,
    param_grid={"model__max_features": mtry},
    scoring="roc_auc",
    n_jobs=-1
)

gridsearch_rf.fit(X_train, y_train)

In [ ]:
print("Beste parameter:", gridsearch_rf.best_params_)
print("Beste AUC:", gridsearch_rf.best_score_)